#  IMDB Sentiment Analysis — v2: BiLSTM + GloVe

**Improvements over v1:**
-  HTML tag removal + negation handling in preprocessing
-  Vocabulary increased from 5,000 → 20,000 words
-  Pretrained GloVe (100d) word embeddings
-  Bidirectional LSTM architecture
-  Early Stopping to prevent overfitting
-  Full evaluation: Accuracy, F1, Precision, Recall, Confusion Matrix

In [ ]:
!pip install kaggle --quiet

In [ ]:
# Install All the necessary libraries & Import Dependencies
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from zipfile import ZipFile
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM,
    Dense, GlobalMaxPooling1D, Dropout
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# load the dataset via kaggle API 
kaggle_dictionary = json.load(open("kaggle.json"))
os.environ["KAGGLE_USERNAME"] = kaggle_dictionary["username"]
os.environ["KAGGLE_KEY"]      = kaggle_dictionary["key"]

!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews --quiet

with ZipFile("imdb-dataset-of-50k-movie-reviews.zip", "r") as z:
    z.extractall()

print("Dataset downloaded and extracted!")

In [ ]:
data = pd.read_csv("/content/IMDB Dataset.csv")
print(f"Dataset shape: {data.shape}")
data.head()

In [ ]:
def clean_text(text):
    
    
    text = re.sub(r'<.*?>', ' ', text)  # this removes any html tag like <.*?>
    
    text = re.sub(r"won't",  "will not", text)
    text = re.sub(r"can't",  "cannot",   text) # this expands the contradictions and handles negations
    text = re.sub(r"n't",    " not",     text)
    text = re.sub(r"'re",    " are",     text)
    text = re.sub(r"'s",     " is",      text)
    text = re.sub(r"'d",     " would",   text)
    text = re.sub(r"'ll",    " will",    text)
    text = re.sub(r"'ve",    " have",    text)
    text = re.sub(r"'m",     " am",      text)
    text = re.sub(r'[^a-zA-Z\s]', '', text) # it removes special characters 

    return text.lower().strip()

print("Before:", data["review"][0][:200])
data["review"] = data["review"].apply(clean_text)
print("\nAfter: ", data["review"][0][:200])

In [ ]:

data["sentiment"] = data["sentiment"].map({"positive": 1, "negative": 0}) #encoding the labels
print(data["sentiment"].value_counts())

In [ ]:

train_data, test_data = train_test_split(data, test_size=0.2, random_state=42) #training and testing the dataset values
print(f"Train size: {train_data.shape[0]} | Test size: {test_data.shape[0]}")

In [ ]:
VOCAB_SIZE  = 20000   # Increased the vocabulary size from 5000 to 20000 compared to the previous version i created.
MAX_LEN     = 200
EMBED_DIM   = 100     # this must match GloVe file

tokenizer = Tokenizer(num_words=VOCAB_SIZE)
tokenizer.fit_on_texts(train_data["review"])

X_train = pad_sequences(
    tokenizer.texts_to_sequences(train_data["review"]), maxlen=MAX_LEN
)
X_test  = pad_sequences(
    tokenizer.texts_to_sequences(test_data["review"]),  maxlen=MAX_LEN
)

Y_train = train_data["sentiment"].values
Y_test  = test_data["sentiment"].values

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")

In [ ]:
# Download GloVe 100d vectors
!wget -q http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip -d glove 
print("GloVe downloaded!") #GloVe pretrained embeddings

In [ ]:
# Parse GloVe file into a dictionary
embeddings_index = {}
with open("glove/glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word   = values[0]
        vector = np.array(values[1:], dtype="float32")
        embeddings_index[word] = vector

print(f"Total GloVe vectors loaded: {len(embeddings_index):,}")

In [ ]:
word_index       = tokenizer.word_index
embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM)) # Build embedding matrix
matched          = 0

for word, idx in word_index.items():
    if idx < VOCAB_SIZE:
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[idx] = vec
            matched += 1

coverage = matched / min(len(word_index), VOCAB_SIZE) * 100
print(f"Words matched with GloVe: {matched:,} ({coverage:.1f}% coverage)")

In [ ]:
model = Sequential([
    # GloVe embedding : frozen so we don't overwrite pretrained weights 
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN,
        weights=[embedding_matrix],
        trainable=False               # freeze GloVe weights
    ),
    # Bidirectional LSTM reads text forwards and backwards
    Bidirectional(LSTM(128, return_sequences=True,
                       dropout=0.3, recurrent_dropout=0.2)),
    GlobalMaxPooling1D(),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, Y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
loss, accuracy = model.evaluate(X_test, Y_test, verbose=0)
print(f"Test Loss:     {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# Full classification report
Y_pred = (model.predict(X_test) > 0.5).astype(int)

print("\n📋 Classification Report:")
print(classification_report(
    Y_test, Y_pred,
    target_names=["Negative", "Positive"]
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(Y_test, Y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)
plt.title("Confusion Matrix — BiLSTM + GloVe", fontsize=14)
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
# Training History Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["accuracy"],     label="Train", linewidth=2)
ax1.plot(history.history["val_accuracy"], label="Validation", linewidth=2)
ax1.set_title("Model Accuracy", fontsize=13)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax2.plot(history.history["loss"],     label="Train", linewidth=2)
ax2.plot(history.history["val_loss"], label="Validation", linewidth=2)
ax2.set_title("Model Loss", fontsize=13)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("BiLSTM + GloVe Training History", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_history.png")

In [ ]:
# Unfreeze embedding layer and train at very low learning rate
# This lets GloVe adapt slightly to IMDB-specific vocabulary

model.layers[0].trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    X_train, Y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

loss_ft, acc_ft = model.evaluate(X_test, Y_test, verbose=0)
print(f"Fine-tuned Test Accuracy: {acc_ft*100:.2f}%")

In [ ]:
def predict_sentiment(review):
    """Predict sentiment for a single review string."""
    cleaned   = clean_text(review)
    sequence  = tokenizer.texts_to_sequences([cleaned])
    padded    = pad_sequences(sequence, maxlen=MAX_LEN)
    prob      = model.predict(padded, verbose=0)[0][0]
    label     = "Positive 😊" if prob > 0.5 else "Negative 😞"
    confidence = prob if prob > 0.5 else 1 - prob
    print(f"Review   : {review}")
    print(f"Sentiment: {label} ({confidence*100:.1f}% confidence)")
    print()

predict_sentiment("This movie was absolutely fantastic! I loved every second of it.")
predict_sentiment("This was the worst film I have ever seen. Complete waste of time.")
predict_sentiment("It was ok, not great but not terrible either.")